Data Source : https://www.kaggle.com/datasets/PromptCloudHQ/flipkart-products


**About Dataset**  
Context  
This is a pre-crawled dataset, taken as subset of a bigger dataset (more than 5.8 million products) that was created by extracting data from Flipkart.com, a leading Indian eCommerce store.  

Content
This dataset has following fields:

* product_url
* product_name
* product_category_tree
* pid
* retail_price
* discounted_price
* image
* is_FK_Advantage_product
* description
* product_rating
* overall_rating
* brand
* product_specifications


#Install and Import Libraries

In [1]:
#download and install langchain dependencies
!pip install -qU langchain-community faiss-cpu langchain-openai

In [2]:
# importing all libraries needed
import pandas as pd #for data processing

import faiss #for faiss vector database
from langchain_community.docstore.in_memory import InMemoryDocstore #for allocating vector database storage (in RAM)
from langchain_community.vectorstores import FAISS #for langchain - faiss vector database connector
from langchain_core.messages import SystemMessage, HumanMessage #for make system prompt and human input question
from langchain_openai import OpenAIEmbeddings, ChatOpenAI #for embedding and LLM service
from uuid import uuid4 #for generate id with uuid format

from langchain_core.documents import Document #for document format that stored to vector database collection

#Input API KEY and Setup LLM Service

In [3]:
import getpass #for input password interface
import os #for read and manage python directory path and environment
from google.colab import userdata

#create textbox for input password
if not os.environ.get("OPENAI_API_KEY"):
  try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  except:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [4]:
#define embedding model and llm from openai provider
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

#Load Vector Database

In [5]:
#define path for save faiss index
path_faiss = "/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 3/Day 10/faiss_index"

#create vector store with saved index
vector_store = FAISS.load_local(
    path_faiss, embeddings, allow_dangerous_deserialization=True
)

#Retrieve Data and insert to Prompt

In [11]:
#try to retrieve data from vector database with similarity_search
retrieve_results = vector_store.similarity_search(
    "Women's Cycling Shorts",
    k=3
)
#show the result
retrieve_results

[Document(id='4044c0ac52c1ee4b28777417651faf42', metadata={'uniq_id': '4044c0ac52c1ee4b28777417651faf42', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts"),
 Document(id='0973b37acd0c664e3de26e97e5571454', metadata={'uniq_id': '0973b37acd0c664e3de26e97e5571454', 'product_name': "Alisha Solid Women's Cycling Shorts"}, page_content="Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Black, Red,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 2 Fabric Cotton Lycra Type Cycling S

In [12]:
#inject retrieval results into system prompt (system message)
system_prompt = SystemMessage(
    content=f"You are a clerk in a clothing store. Your job is to respond to user questions related to clothing.\nYour are provided with relevan informations below to answer the question.\nRelevant Information:\n{retrieve_results}"
)
#define user question on human prompt (human message)
human_message = HumanMessage(
    content="I need Women's Cycling Shorts product in your store."
)
response = llm.invoke([system_prompt, human_message]) #generate response form llm
print(response.content) #show llm response


We have several options for Women's Cycling Shorts in our store. Here are a few:

1. **Alisha Solid Women's Cycling Shorts (Pack of 4)**:
   - Fabric: Cotton Lycra
   - Colors Available: White, Black, Red
   - Pattern: Solid
   - Care: Gentle Machine Wash in Lukewarm Water, Do Not Bleach
   - Style Code: ALTGHT4P_39

2. **Alisha Solid Women's Cycling Shorts (Pack of 2)**:
   - Fabric: Cotton Lycra
   - Colors Available: Black, Red
   - Pattern: Solid
   - Care: Gentle Machine Wash in Lukewarm Water, Do Not Bleach
   - Style Code: ALTGHT_11

3. **Alisha Solid Women's Cycling Shorts (Pack of 4)**:
   - Fabric: Cotton Lycra
   - Colors Available: Navy, Red, White
   - Pattern: Solid
   - Care: Gentle Machine Wash in Lukewarm Water, Do Not Bleach
   - Style Code: ALTGHT4P_26

If you have a preference regarding the pack size or color, let me know, and I can provide more details!


In [9]:
#inject retrieval results into system prompt (system message)
system_prompt = SystemMessage(
    content=f"You are a clerk in a clothing store. Your job is to respond to user questions related to clothing.\nYour are provided with relevan informations below to answer the question.\nRelevant Information:\n{retrieve_results}"
)
#define user question on human prompt (human message)
human_message = HumanMessage(
    content="Hello how are you?"
)
response = llm.invoke([system_prompt, human_message]) #generate response form llm
print(response.content) #show llm response


Hello! I'm here to help you with any questions you have about clothing. How can I assist you today?


In [10]:
system_prompt

SystemMessage(content='You are a clerk in a clothing store. Your job is to respond to user questions related to clothing.\nYour are provided with relevan informations below to answer the question.\nRelevant Information:\n[Document(id=\'4044c0ac52c1ee4b28777417651faf42\', metadata={\'uniq_id\': \'4044c0ac52c1ee4b28777417651faf42\', \'product_name\': "Alisha Solid Women\'s Cycling Shorts"}, page_content="Key Features of Alisha Solid Women\'s Cycling Shorts Cotton Lycra White, Black, Red, Black,Specifications of Alisha Solid Women\'s Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 4 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women\'s Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTGHT4P_39 In the Box 4 shorts"), Document(id=\'0973b37acd0c664e3de26e97e5571454\', metadata={\'uniq_id\': \'0973b37acd0c664e3de26e97e5571454\', \'product_name\': "Alisha Solid Women\'s Cycling Shorts"}